### PulmoTriage · Patient Summary Generator

#### Install

In [ ]:
!pip install -q captum medspacy

#### Input files

In [ ]:
import os, glob

def find_file(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return hits[0] if hits else None

In [ ]:
DATA_CSV     = find_file('nlp_dataset.csv')
CLASSES_JSON = find_file('mlb_classes.json')
PRED_CSV     = find_file('text_predictions.csv')
_meta        = find_file('text_model/meta.json')
MODEL_DIR    = os.path.dirname(_meta) if _meta else None
TEXT_MODULE  = find_file('pulmo_text.py')
GOLD_CSV     = 'extractor_gold.csv'
OUT_DIR      = '/kaggle/working'
assert DATA_CSV and PRED_CSV, 'Add nlp_dataset.csv and text_predictions.csv as inputs'
print(DATA_CSV, PRED_CSV, MODEL_DIR or '(text model folder not attached: highlighted words skipped)', sep='\n')

#### Settings

In [ ]:
SEED = 42
NER_MODEL = 'd4data/biomedical-ner-all'
N_ANNOTATE = 60
RUN_LLM = True
LLM_NAME = 'google/flan-t5-base'
N_LLM = 30
N_RATING = 30

#### Imports

In [ ]:
import sys, json, re, shutil, zipfile, random
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
warnings.filterwarnings('ignore')
random.seed(SEED); np.random.seed(SEED)
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)

### Summary module (`pulmo_summary.py`)

In [ ]:
%%writefile pulmo_summary.py
import re

SYMPTOMS = {
    "chest pain":           [r"chest pain", r"\bcp\b", r"chest discomfort", r"chest tightness", r"chest pressure"],
    "shortness of breath":  [r"shortness of breath", r"short of breath", r"\bsob\b", r"dyspn(?:o)?ea", r"difficulty breathing",
                             r"breathless\w*", r"\bdoe\b"],
    "cough":                [r"\bcough(?:ing|s)?\b"],
    "sputum":               [r"sputum", r"productive"],
    "fever":                [r"fevers?\b", r"febrile", r"pyrexia"],
    "chills":               [r"chills"],
    "coughing blood":       [r"ha?emoptysis", r"coughing up blood"],
    "wheezing":             [r"wheez\w*"],
    "dizziness / fainting": [r"syncop\w*", r"presyncop\w*", r"dizz\w*", r"lightheaded\w*", r"passed out"],
    "fatigue / weakness":   [r"fatigue", r"weakness", r"malaise", r"lethargy"],
    "weight loss":          [r"weight loss"],
    "palpitations":         [r"palpitations?", r"tachycardi\w*"],
    "low oxygen":           [r"hypoxi\w*", r"desaturation", r"low oxygen"],
    "swelling":             [r"o?edema", r"swelling"],
    "abdominal pain":       [r"abdominal pain", r"\babd pain\b", r"epigastric pain"],
    "back / rib pain":      [r"back pain", r"rib pain", r"flank pain", r"shoulder pain"],
    "nausea / vomiting":    [r"nausea", r"vomit\w*"],
}
HISTORY = {
    "smoker":                 [r"smok\w*", r"tobacco", r"pack years?"],
    "COPD / emphysema":       [r"\bcopd\b", r"emphysema"],
    "asthma":                 [r"asthma\w*"],
    "cancer":                 [r"cancer", r"carcinoma", r"malignan\w*", r"tumou?r", r"lymphoma", r"metasta\w*"],
    "tuberculosis / TB test": [r"\btb\b", r"tubercul\w*", r"\bppd\b", r"quantiferon"],
    "heart failure":          [r"\bchf\b", r"heart failure", r"cardiomyopathy"],
    "coronary disease":       [r"\bcad\b", r"coronary", r"myocardial infarction", r"\bcabg\b", r"\bstemi\b"],
    "high blood pressure":    [r"hypertension", r"\bhtn\b"],
    "diabetes":               [r"diabet\w*", r"\bdm\b"],
    "pneumonia":              [r"pneumonia", r"\bpna\b"],
    "blood clot (PE / DVT)":  [r"pulmonary embol\w*", r"\bdvt\b", r"\bpe\b"],
    "HIV":                    [r"\bhiv\b"],
    "trauma / fall":          [r"trauma\w*", r"\bfall\b", r"\bfell\b", r"\bmva\b", r"\bmvc\b", r"accident", r"injur\w*", r"gunshot"],
}
PURPOSE = {
    "pre-operative check":  [r"pre[- ]?op\w*", r"before surgery", r"surgical clearance"],
    "routine / screening":  [r"routine", r"screening", r"\bphysical\b", r"annual", r"employment", r"immigration", r"check ?up"],
    "follow-up":            [r"follow[- ]?up", r"\bf/?u\b", r"\bre-?eval\w*"],
    "line / tube position": [r"line placement", r"tube placement", r"\bpicc\b", r"\bport placement\b", r"\bng tube\b", r"\bet tube\b"],
}
LEXICON = {"SYMPTOM": SYMPTOMS, "CONDITION": HISTORY, "PURPOSE": PURPOSE}

NER_MODEL = "d4data/biomedical-ner-all"
NER_TYPES = {"Sign_symptom": "SYMPTOM", "Disease_disorder": "CONDITION", "History": "CONDITION", "Family_history": "CONDITION",
             "Diagnostic_procedure": "OTHER", "Therapeutic_procedure": "OTHER", "Medication": "OTHER",
             "Duration": "DURATION", "Age": "AGE", "Sex": "SEX"}
NER_MODIFIERS = {"Biological_structure", "Severity", "Detailed_description", "Qualitative_concept"}
HEADS = {"Sign_symptom", "Disease_disorder"}
NER_MIN_SCORE = 0.5

CONTEXT_RULES = [
    *[(c, "POSSIBLE_EXISTENCE", "FORWARD") for c in ["evaluate for", "eval for", "evaluation for", "assess for", "assessment for",
                                                     "concern for", "question of", "suspected", "suspect", "possible", "probable"]],
    *[(c, "HISTORICAL", "FORWARD") for c in ["history of", "hx", "hx of", "h/o", "previous", "prior", "past", "known",
                                             "status post", "s/p"]],
    *[(c, "FAMILY", "FORWARD") for c in ["family history of", "fhx", "mother", "father", "sister", "brother"]],
    *[(c, "TERMINATE", "TERMINATE") for c in [",", ";", "but", "however", "now", "with", "presents with", "presenting with",
                                              "complains of", "c/o", "currently", "positive"]],
]
CONTEXT_REMOVE = {"evaluate for", "checked for", "positive", "presentation", "comes in with"}
CONTEXT_MAX_SCOPE = 6
_CATS = ["NEGATED_EXISTENCE", "POSSIBLE_EXISTENCE", "HISTORICAL", "HYPOTHETICAL", "FAMILY"]
CONTEXT_TERMINATING = {c: [o for o in _CATS if o != c] for c in _CATS}

_NUM = r"(?:\d+|one|two|three|four|five|six|seven|eight|nine|ten|several|few|a|1-2|2-3)"
_UNIT = r"(?:hours?|hrs?|days?|weeks?|wks?|months?|mos?|years?|yrs?)"
DURATION = re.compile(rf"\b(?:for|x|since|past|over|last)\s*(?:the\s+)?({_NUM}\s*{_UNIT})\b|\b({_NUM}\s*{_UNIT})\s+ago\b")
NUM_UNIT = re.compile(rf"\b({_NUM}\s*{_UNIT})\b")
AGE = re.compile(r"\b(\d{1,3})[- ]?(?:years?|yrs?|yo|y/o)[- ]?(?:old)?\b")
SEX = {"male": [r"\bmale\b", r"\bman\b", r"\bgentleman\b", r"\bboy\b", r"\bm\b"],
       "female": [r"\bfemale\b", r"\bwoman\b", r"\blady\b", r"\bgirl\b", r"\bf\b"]}
NOISE = re.compile(r"hidden|xxxx|^\W*$")
SEX_WORDS = {"male", "female", "man", "woman", "gentleman", "lady", "boy", "girl"}
GENERIC = {"evaluation", "eval", "evaluate", "status", "pt", "patient", "history", "hx", "exam", "examination", "workup",
           "check", "symptoms", "complaints", "problem", "problems", "disease", "condition", "loss", "sore", "onset", "shielded",
           "asymptomatic"}

PRETTY = {"lung": "lung abnormality", "opacity": "lung opacity", "markings": "prominent lung markings",
          "density": "increased density", "spine": "spine changes", "thoracic vertebrae": "thoracic spine changes",
          "technical quality of image unsatisfactory": "poor image quality", "catheters, indwelling": "indwelling catheter",
          "fractures, bone": "bone fracture", "aorta": "aortic changes", "diaphragm": "diaphragm changes",
          "cicatrix": "scarring", "calcinosis": "calcification", "pulmonary atelectasis": "atelectasis"}
SKIP_LABELS = {"no indexing"}

def pretty(label):
    return PRETTY.get(label, label)

_STATE = {"nlp": None, "matcher": None, "context": None, "ner": None, "method": "rules", "cache": {}}

def _pipeline():
    if _STATE["nlp"] is None:
        import spacy
        from medspacy.target_matcher import TargetMatcher, TargetRule
        from medspacy.context import ConText, ConTextRule
        nlp = spacy.blank("en")
        nlp.add_pipe("sentencizer")
        matcher = TargetMatcher(nlp)
        matcher.add([TargetRule(concept, etype, pattern="|".join(pats))
                     for etype, group in LEXICON.items() for concept, pats in group.items()])
        default = [r for r in ConText(nlp, rules="default").rules if r.literal not in CONTEXT_REMOVE]
        context = ConText(nlp, rules=None, max_scope=CONTEXT_MAX_SCOPE, terminating_types=CONTEXT_TERMINATING)
        context.add(default + [ConTextRule(lit, cat, direction=d) for lit, cat, d in CONTEXT_RULES])
        _STATE.update(nlp=nlp, matcher=matcher, context=context)
    return _STATE["nlp"], _STATE["matcher"], _STATE["context"]

def load_ner(model=NER_MODEL, device=None):
    try:
        if callable(model):
            _STATE["ner"] = model
        else:
            import torch
            from transformers import pipeline
            dev = device if device is not None else (0 if torch.cuda.is_available() else -1)
            _STATE["ner"] = pipeline("token-classification", model=model, aggregation_strategy="simple", device=dev)
        _STATE["method"], _STATE["cache"] = "hybrid", {}
        return True
    except Exception as e:
        print(f"NER model not loaded ({type(e).__name__}: {e}); using the lexicon matcher.")
        return False

def set_method(method):
    assert method in ("rules", "ner", "hybrid")
    if method != "rules" and _STATE["ner"] is None:
        raise ValueError("load the NER model first: load_ner()")
    _STATE["method"] = method

def get_method():
    return _STATE["method"]

def _ner_entities(text):
    if text not in _STATE["cache"]:
        raw = [e for e in sorted(_STATE["ner"](text), key=lambda e: e["start"]) if float(e["score"]) >= NER_MIN_SCORE]
        touching = lambda a, b: not text[a["end"]:b["start"]].strip()
        out, pending = [], None
        for e in raw:
            label = e["entity_group"]
            if label in NER_MODIFIERS:
                pending = (pending[0] if pending and touching({"end": pending[1]}, e) else e["start"], e["end"])
                continue
            start = e["start"]
            if pending and label in HEADS and touching({"end": pending[1]}, e):
                start = pending[0]
            pending = None
            same_kind = out and NER_TYPES.get(out[-1][2]) == NER_TYPES.get(label) in ("SYMPTOM", "CONDITION")
            if same_kind and touching({"end": out[-1][1]}, e):
                out[-1] = (out[-1][0], e["end"], out[-1][2])
            elif label in NER_TYPES:
                out.append((start, e["end"], label))
        _STATE["cache"][text] = out
    return _STATE["cache"][text]

def _normalise(span_text, etype):
    low = re.sub(r"\s+", " ", span_text.lower()).strip(" -/")
    for t in [etype] + [x for x in LEXICON if x != etype]:
        hits = [c for c, pats in LEXICON.get(t, {}).items() if any(re.search(p, low) for p in pats)]
        if hits:
            return hits, t
    if low in GENERIC or len(low) < 3:
        return [], etype
    return [low], etype

def _trim_noise(span):
    bad = lambda t: NOISE.search(t.text.lower()) or (span.label_ != "SEX" and t.text.lower() in SEX_WORDS)
    keep, best = [], []
    for tok in span:
        if bad(tok):
            keep = []
        else:
            keep.append(tok)
            best = list(keep)
    while best and (best[0].is_stop or best[0].is_punct):
        best = best[1:]
    while best and (best[-1].is_stop or best[-1].is_punct):
        best = best[:-1]
    if not best:
        return None
    return span.doc.char_span(best[0].idx, best[-1].idx + len(best[-1].text), label=span.label_)

def analyze(note, method=None):
    from spacy.util import filter_spans
    nlp, matcher, context = _pipeline()
    method = method or _STATE["method"]
    doc = nlp(str(note))
    spans = list(matcher(nlp(str(note))).ents) if method in ("rules", "hybrid") else []
    spans = [doc.char_span(s.start_char, s.end_char, label=s.label_) for s in spans]
    if method in ("ner", "hybrid") and _STATE["ner"] is not None:
        for start, end, label in _ner_entities(str(note)):
            s = doc.char_span(start, end, label=NER_TYPES[label], alignment_mode="expand")
            if s is not None:
                s = _trim_noise(s)
            if s is not None:
                spans.append(s)
    doc.ents = filter_spans([s for s in spans if s is not None])
    return context(doc)

def extract_clinical_info(note, method=None):
    info = {"symptoms": [], "denied": [], "history": [], "purpose": [], "other": [], "duration": None, "age": None, "sex": None,
            "entities": []}
    if not note or not str(note).strip():
        return info
    add = lambda key, val: info[key].append(val) if val not in info[key] else None
    doc = analyze(note, method)
    for ent in doc.ents:
        etype = ent.label_
        if etype in ("DURATION", "AGE", "SEX"):
            low = ent.text.lower()
            if etype == "DURATION" and info["duration"] is None and NUM_UNIT.search(low):
                info["duration"] = NUM_UNIT.search(low).group(1)
            elif etype == "AGE" and info["age"] is None and re.search(r"\d+", low):
                info["age"] = int(re.search(r"\d+", low).group())
            elif etype == "SEX":
                info["sex"] = next((s for s, pats in SEX.items() if any(re.search(p, low) for p in pats)), info["sex"])
            continue
        concepts, etype = _normalise(ent.text, etype)
        if ent._.is_negated:
            status = "denied"
        elif ent._.is_family:
            status = "family"
        elif ent._.is_uncertain or ent._.is_hypothetical:
            status = "possible"
        elif ent._.is_historical:
            status = "historical"
        else:
            status = "present"
        for c in concepts:
            info["entities"].append({"text": ent.text, "concept": c, "type": etype, "status": status})
            if etype == "PURPOSE":
                add("purpose", c)
            elif status == "denied":
                add("denied", c)
            elif status == "family":
                add("history", f"family history of {c}")
            elif status == "possible":
                add("purpose", f"possible {c}")
            elif etype == "OTHER":
                add("other", c)
            elif etype == "SYMPTOM" and status == "historical":
                add("history", f"previous {c}")
            elif etype == "SYMPTOM":
                add("symptoms", c)
            else:
                add("history", c)

    low = " " + str(note).lower() + " "
    if info["duration"] is None and DURATION.search(low):
        m = DURATION.search(low)
        info["duration"] = (m.group(1) or m.group(2)).strip()
    if info["age"] is None and AGE.search(low):
        info["age"] = int(AGE.search(low).group(1))
    if info["sex"] is None:
        info["sex"] = next((s for s, pats in SEX.items() if any(re.search(p, low) for p in pats[:-1])), None)
    if "sputum" in info["symptoms"] and "cough" in info["symptoms"]:
        info["symptoms"] = [s if s != "cough" else "cough with sputum" for s in info["symptoms"] if s != "sputum"]
    return info

def _join(items):
    items = list(items)
    if len(items) <= 1:
        return "".join(items)
    return ", ".join(items[:-1]) + " and " + items[-1]

def _findings_list(result, k=3):
    if not result:
        return []
    items = result.get("findings") or result.get("top") or []
    items = [d for d in items if d["label"] not in SKIP_LABELS and d["label"] != "normal"]
    return items[:k]

NEXT_STEP = {
    "high":   "Review first: radiologist read as soon as possible.",
    "medium": "Review today in the normal queue.",
    "low":    "Routine review.",
}
ORDER = ["presentation", "reason", "history", "denied", "other", "assessment", "image", "note_model", "key_words", "priority",
         "disclaimer"]

def make_summary(note, text_result=None, image_result=None, fusion_result=None, priority=None, key_words=None):
    info = extract_clinical_info(note)
    sections, facts = {}, {"note_terms": {k: v for k, v in info.items() if k != "entities"}, "findings": []}

    who = {"male": "male patient", "female": "female patient"}.get(info["sex"], "patient")
    who = f"{info['age']}-year-old {who}" if info["age"] else who.capitalize()
    if info["symptoms"]:
        dur = f" for {info['duration']}" if info["duration"] else ""
        sections["presentation"] = f"{who} presenting with {_join(info['symptoms'])}{dur}."
    else:
        sections["presentation"] = f"{who} with " + ("no symptoms stated in the note." if note and str(note).strip()
                                                      else "no clinical note.")
    if info["purpose"]:
        sections["reason"] = f"Reason for exam: {_join(info['purpose'])}."
    if info["history"]:
        sections["history"] = f"History: {_join(info['history'])}."
    if info["denied"]:
        sections["denied"] = f"Denies / no: {_join(info['denied'])}."
    if info["other"]:
        sections["other"] = f"Also mentioned: {_join(info['other'])}."

    def describe(name, result, extra=""):
        items = _findings_list(result)
        facts["findings"] += [d["label"] for d in items]
        if not items:
            return f"{name}: no specific finding stands out{extra}."
        listed = _join(f"{pretty(d['label'])} ({d['probability']:.0%})" for d in items)
        return f"{name}: most likely {listed}{extra}."

    if fusion_result:
        has_note = bool(note and str(note).strip())
        sections["assessment"] = describe("Combined image + note assessment" if has_note else "Combined assessment", fusion_result)
    if image_result:
        region = f", mainly in the {image_result['region']}" if image_result.get("region") else ""
        sections["image"] = describe("Image analysis", image_result, region)
    if text_result and text_result.get("has_note"):
        pn = text_result.get("p_normal")
        normal = f" Probability of a normal study from the note alone: {pn:.0%}." if pn is not None else ""
        sections["note_model"] = describe("From the note alone", text_result) + normal
        if key_words:
            sections["key_words"] = f"Words that influenced the note analysis: {', '.join(key_words[:5])}."
    elif text_result is not None:
        sections["note_model"] = "Note analysis skipped: no usable clinical note."

    if priority:
        sections["priority"] = f"Priority: {priority.upper()}. {NEXT_STEP.get(priority.lower(), '')}".strip()
    elif not (image_result or fusion_result):
        sections["priority"] = "Priority: pending the image analysis."

    sections["disclaimer"] = "AI-assisted summary (educational prototype). The final decision is the doctor's."
    text = " ".join(sections[k] for k in ORDER if k in sections)
    return {"sections": sections, "text": text, "facts": facts}

def check_summary(summary, note, text_result=None, image_result=None, fusion_result=None):
    problems = []
    info = extract_clinical_info(note)
    for key in ("symptoms", "history", "denied", "purpose", "other"):
        for term in summary["facts"]["note_terms"].get(key, []):
            if term not in info[key]:
                problems.append(f"'{term}' is not in the note")
    allowed = set()
    for r in (text_result, image_result, fusion_result):
        if r:
            allowed |= {d["label"] for d in (r.get("findings") or []) + (r.get("top") or [])}
    for label in summary["facts"]["findings"]:
        if label not in allowed:
            problems.append(f"finding '{label}' was not predicted by any model")
    for term in info["denied"]:
        if term in info["symptoms"]:
            problems.append(f"'{term}' listed as both present and denied")
    return problems

#### Load the module and the NER model

In [ ]:
import importlib, pulmo_summary as ps
importlib.reload(ps)
try:
    from loguru import logger; logger.remove()
except ImportError:
    pass
HAS_NER = ps.load_ner(NER_MODEL)
print('NER model loaded:', HAS_NER, '| extractor:', ps.get_method())

#### Example: what the pipeline finds

In [ ]:
note = '65 year old man with dyspnea and productive cough for 3 days, history of COPD, evaluate for pneumonia, no fever'
doc = ps.analyze(note)
pd.DataFrame([{'text': e.text, 'type': e.label_, 'negated': e._.is_negated, 'historical': e._.is_historical,
               'possible': e._.is_uncertain, 'family': e._.is_family} for e in doc.ents])

#### Highlighted view (entities and their ConText cues)

In [ ]:
from medspacy.visualization import visualize_ent
visualize_ent(doc)

#### Structured result

In [ ]:
{k: v for k, v in ps.extract_clinical_info(note).items() if v and k != 'entities'}

### Which extractor is best? (gold-standard evaluation)

Three extractors are compared on the same annotated notes:

#### Gold-standard notes (a data file: one annotated note per row)

In [ ]:
%%writefile extractor_gold.csv
note,symptoms,denied,history,purpose,duration,age,sex
Chest pain,chest pain,,,,,,
chest pain,chest pain,,,,,,
dyspnea,shortness of breath,,,,,,
SOB,shortness of breath,,,,,,
shortness of breath for 3 days,shortness of breath,,,,3 days,,
"XXXX-year-old male with chest pain and SOB for 2 weeks, no fever.",chest pain; shortness of breath,fever,,,2 weeks,,male
hiddenyearold female with productive cough x3 days denies hemoptysis,cough with sputum,coughing blood,,,3 days,,female
Positive PPD,,,tuberculosis / TB test,,,,
positive tb test,,,tuberculosis / TB test,,,,
Pre-op,,,,pre-operative check,,,
preop bariatric surgery,,,,pre-operative check,,,
Pre-operative evaluation,,,,pre-operative check,,,
routine physical,,,,routine / screening,,,
Immigration screening,,,,routine / screening,,,
follow-up pneumonia,,,pneumonia,follow-up,,,
"History of COPD, smoker, now with wheezing",wheezing,,COPD / emphysema; smoker,,,,
history of lung cancer,,,cancer,,,,
Fever and cough,fever; cough,,,,,,
"No fever or chills, cough",cough,fever; chills,,,,,
denies chest pain,,chest pain,,,,,
without shortness of breath,,shortness of breath,,,,,
syncope,dizziness / fainting,,,,,,
lightheadedness today,dizziness / fainting,,,,,,
hemoptysis,coughing blood,,,,,,
"fall, rib pain",back / rib pain,,trauma / fall,,,,
"MVA, chest wall pain",chest wall pain,,trauma / fall,,,,
"CHF exacerbation, edema",swelling,,heart failure,,,,
"HTN, diabetes, chest pain",chest pain,,high blood pressure; diabetes,,,,
"history of pneumonia, now with fever",fever,,pneumonia,,,,
"previous chest pain, now asymptomatic",,,previous chest pain,,,,
hypoxia,low oxygen,,,,,,
palpitations,palpitations,,,,,,
weight loss and fatigue for 2 months,weight loss; fatigue / weakness,,,,2 months,,
PICC line placement,,,,line / tube position,,,
asthma exacerbation,,,asthma,,,,
HIV positive with cough,cough,,HIV,,,,
"nausea and vomiting, abdominal pain",nausea / vomiting; abdominal pain,,,,,,
65 year old man with dyspnea,shortness of breath,,,,,65,male
XXXX XXXX,,,,,,,
,,,,,,,
night sweats and cough for 3 weeks,night sweats; cough,,,,3 weeks,,
stridor,stridor,,,,,,
hoarseness for 2 months,hoarseness,,,,2 months,,
leg swelling and dyspnea,swelling; shortness of breath,,,,,,
Evaluate for pneumonia,,,,possible pneumonia,,,
rule out pneumothorax,,,,possible pneumothorax,,,
family history of lung cancer,,,family history of cancer,,,,
sarcoidosis,,,sarcoidosis,,,,
pulmonary fibrosis follow-up,,,pulmonary fibrosis,follow-up,,,
denies chest pain shortness of breath,,chest pain; shortness of breath,,,,,
history of copd now with dyspnea,shortness of breath,,COPD / emphysema,,,,
chest pain no fever history of copd,chest pain,fever,COPD / emphysema,,,,
sickle cell disease with chest pain,chest pain,,sickle cell disease,,,,
lupus with pleuritic chest pain,chest pain,,lupus,,,,
45 year old female with headache,headache,,,,,45,female
dysphagia,dysphagia,,,,,,
Pleural effusion follow up,,,pleural effusion,follow-up,,,
hiddenyearold male smoker with chronic cough,cough,,smoker,,,,male
sepsis,,,sepsis,,,,
hypertension no chest pain,,chest pain,high blood pressure,,,,

#### Scoring function

In [ ]:
KEYS = ['symptoms', 'denied', 'history', 'purpose']

def same(a, b):
    a, b = a.lower(), b.lower()
    return a == b or (min(len(a), len(b)) >= 4 and (a in b or b in a))

In [ ]:
def score_extractor(gold, method):
    tp = fp = fn = 0
    rows = []
    for g in gold.itertuples():
        got = ps.extract_clinical_info(g.note, method)
        ok = True
        for k in KEYS:
            exp = [x.strip() for x in getattr(g, k).split(';') if x.strip()]
            hit = sum(any(same(p, e) for e in exp) for p in got[k])
            tp += hit; fp += len(got[k]) - hit
            fn += sum(not any(same(p, e) for p in got[k]) for e in exp)
            ok &= hit == len(got[k]) == len(exp)
        ok &= got['duration'] == (g.duration or None) and got['sex'] == (g.sex or None)
        ok &= got['age'] == (int(g.age) if g.age else None)
        rows.append({'note': g.note, 'correct': ok, **{k: got[k] for k in KEYS + ['duration', 'age', 'sex']}})
    p, r = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {'fully correct': np.mean([x['correct'] for x in rows]), 'precision': p, 'recall': r,
            'F1': 2 * p * r / max(p + r, 1e-9)}, pd.DataFrame(rows)

In [ ]:
def compare(gold):
    results, details = {}, {}
    for m in METHODS:
        results[m], details[m] = score_extractor(gold, m)
    return pd.DataFrame(results).T, details

#### Compare the extractors

In [ ]:
METHODS = ['rules', 'ner', 'hybrid'] if HAS_NER else ['rules']
gold = pd.read_csv(GOLD_CSV, keep_default_na=False, dtype=str)
comparison, details = compare(gold)
print(comparison.round(3).to_string())
display(comparison.style.format('{:.1%}').highlight_max(color='#d1fae5'))

#### Use the best extractor

In [ ]:
BEST = comparison['F1'].idxmax()
ps.set_method(BEST)
print('Extractor used from now on:', BEST)
errors = details[BEST][~details[BEST].correct]
print(f'{len(errors)} of {len(gold)} gold notes not fully correct')
errors

### Extraction on the real notes

#### Load notes and text-model predictions

In [ ]:
data = pd.read_csv(DATA_CSV, keep_default_na=False, na_values=['']).drop_duplicates('uid')
pred = pd.read_csv(PRED_CSV)
CLASSES = [c[2:] for c in pred.columns if c.startswith('p_')]
df = data[['uid', 'split', 'indication']].merge(pred, on=['uid', 'split'], how='inner').copy()
df['note'] = df['indication'].replace('not provided', '')
has_note = df.note != ''
dev_notes = df.loc[has_note & (df.split != 'test'), 'note'].drop_duplicates()
test_notes = df.loc[has_note & (df.split == 'test'), 'note'].drop_duplicates()
print(len(df), 'patients |', has_note.sum(), 'with a note |', len(dev_notes), 'different development notes |', len(test_notes), 'different test notes')

#### Error analysis (development notes): terms the NER model found that are not in the lexicon

In [ ]:
if HAS_NER:
    ents = pd.DataFrame([e for n in dev_notes for e in ps.extract_clinical_info(n, 'hybrid')['entities']])
    lexicon = {c for g in ps.LEXICON.values() for c in g}
    new_terms = ents[~ents.concept.isin(lexicon)].groupby(['concept', 'type']).size().sort_values(ascending=False)
    print(f'{len(new_terms)} different terms found only by the NER model, e.g.:')
    display(new_terms.head(15).to_frame('notes'))

#### Error analysis (development notes): notes where nothing was extracted

In [ ]:
found_dev = dev_notes.apply(lambda n: ps.extract_clinical_info(n))
dev_notes[~found_dev.apply(lambda d: any(d[k] for k in KEYS + ['other'])).values].value_counts().head(15)

#### Coverage on the test notes (notes where at least one item is found)

In [ ]:
coverage = {}
for m in METHODS:
    found = test_notes.apply(lambda n: ps.extract_clinical_info(n, m))
    coverage[m] = found.apply(lambda d: any(d[k] for k in KEYS + ['other'])).mean()
coverage = pd.Series(coverage, name='test notes with an extracted item')
print(coverage.round(3).to_string())

#### Extract every note with the chosen method

In [ ]:
info = df['note'].apply(ps.extract_clinical_info)
for k in KEYS + ['other']:
    df[k] = info.apply(lambda d: d[k])
df['duration'] = info.apply(lambda d: d['duration'])
t = df[has_note & (df.split == 'test')]
print(f'Test notes with a negation (e.g. "no fever"): {(t.denied.str.len() > 0).mean():.1%} | with a duration: {t.duration.notna().mean():.1%}')

#### Most frequent items

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for a, k, col in zip(ax, ['symptoms', 'history', 'purpose'], ['#1428A0', '#7c3aed', '#0891b2']):
    s = df[k].explode().dropna().value_counts().head(10).iloc[::-1]
    a.barh(s.index, s.values, color=col); a.set_title(f'{k} (notes)')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/figures/extracted_items.png', dpi=120); plt.show()

### Independent test: team-annotated real notes

`extractor_annotation_sheet.csv` holds real **test** notes with **empty** columns (annotators must not see the extractor's output, to avoid bias).

#### Blank annotation sheet (real test notes)

In [ ]:
sheet = pd.DataFrame({'note': test_notes.sample(min(N_ANNOTATE, len(test_notes)), random_state=SEED).values})
for col in KEYS + ['duration', 'age', 'sex']:
    sheet[col] = ''
sheet.to_csv(f'{OUT_DIR}/extractor_annotation_sheet.csv', index=False)
print('Saved extractor_annotation_sheet.csv with', len(sheet), 'real test notes')

#### Score on the annotated real notes (when attached)

In [ ]:
ANNOTATED_CSV = find_file('extractor_annotated.csv')
if ANNOTATED_CSV:
    annotated = pd.read_csv(ANNOTATED_CSV, keep_default_na=False, dtype=str)
    independent, _ = compare(annotated)
    print(f'Independent test on {len(annotated)} real notes:')
    print(independent.round(3).to_string())
else:
    independent = None
    print('extractor_annotated.csv not attached yet: independent test skipped')

### Summaries for the test patients

The text-model result for each patient is rebuilt from `text_predictions.csv` in the same format the app gets from `TextModel.predict_text()`.

#### Text-model result per patient (same format as TextModel.predict_text)

In [ ]:
thresholds = None
if MODEL_DIR:
    thresholds = np.array(json.load(open(f'{MODEL_DIR}/meta.json'))['thresholds'])

def text_result(row):
    if not row['note']:
        return {'has_note': False, 'findings': [], 'top': []}
    p = np.array([row[f'p_{c}'] for c in CLASSES], dtype=float)
    order = np.argsort(-p)
    item = lambda i: {'label': CLASSES[i], 'probability': round(float(p[i]), 4)}
    res = {'has_note': True, 'top': [item(i) for i in order[:5]],
           'findings': [item(i) for i in order if thresholds is not None and p[i] > thresholds[i]]}
    if 'normal' in CLASSES:
        res['p_normal'] = round(float(p[CLASSES.index('normal')]), 4)
    return res

In [ ]:
test = df[df.split == 'test'].reset_index(drop=True)
test['text_result'] = test.apply(text_result, axis=1)

#### Highlighted words (only if the text model folder is attached)

In [ ]:
tm = None
if MODEL_DIR and TEXT_MODULE:
    sys.path.insert(0, os.path.dirname(TEXT_MODULE))
    import pulmo_text
    tm = pulmo_text.TextModel(MODEL_DIR)
def key_words(note, n=5):
    if tm is None or not note:
        return None
    return [d['word'] for d in tm.explain(note, n_steps=30)['top'] if d['score'] > 0][:n]
print('Highlighted words:', 'on' if tm else 'off')

#### Build every summary

In [ ]:
test['summary'] = [ps.make_summary(r.note, r.text_result) for r in test.itertuples()]
test['summary_text'] = test.summary.apply(lambda s: s['text'])
words = test.summary_text.str.split().str.len()
print(f'{len(test)} summaries | words per summary: median {words.median():.0f}, max {words.max()}')

#### Example summaries

In [ ]:
def card(title, s):
    sec = s['sections']
    lines = ''.join(f'<div style="margin:2px 0">{sec[k]}</div>' for k in ps.ORDER[:-1] if k in sec)
    return (f'<div style="border:1px solid #ccd;border-radius:8px;padding:10px 14px;margin:8px 0;font-family:sans-serif;font-size:13px">'
            f'<div style="color:#1428A0;font-weight:600;margin-bottom:4px">{title}</div>{lines}'
            f'<div style="color:#888;font-size:11px;margin-top:4px">{sec["disclaimer"]}</div></div>')

examples = test[test.note != ''].sample(5, random_state=SEED)
html = ''
for r in examples.itertuples():
    s = ps.make_summary(r.note, r.text_result, key_words=key_words(r.note))
    html += card(f'Patient {r.uid} · note: "{r.note}"', s)
display(HTML(html))

### What the app will show with all models

In the app the summary also includes the image result, the combined (fusion) result and the priority.

In [ ]:
r = examples.iloc[0]
demo_image  = {'findings': [{'label': 'opacity', 'probability': 0.71}, {'label': 'pleural effusion', 'probability': 0.44}], 'region': 'right lower zone'}
demo_fusion = {'findings': [{'label': 'opacity', 'probability': 0.78}]}
display(HTML(card('Example with image + fusion results (example values)',
                  ps.make_summary(r.note, r.text_result, demo_image, demo_fusion, priority='high', key_words=key_words(r.note)))))

### Safety checks

Every summary is checked: each symptom / history item must be in the note, each finding must come from a model output, and nothing may be both present and denied.

#### Check every test summary

In [ ]:
test['problems'] = [ps.check_summary(r.summary, r.note, r.text_result) for r in test.itertuples()]
print(f'Summaries with no problem: {(test.problems.str.len() == 0).mean():.1%} of {len(test)}')

#### Does the checker catch errors? (inject one invented fact into each summary)

In [ ]:
import copy
caught = 0
for r in test.itertuples():
    s = copy.deepcopy(r.summary)
    if random.random() < 0.5:
        s['facts']['note_terms']['symptoms'].append('coughing blood' if 'coughing blood' not in ps.extract_clinical_info(r.note)['symptoms'] else 'fever')
    else:
        s['facts']['findings'].append('tuberculosis' if 'tuberculosis' in CLASSES else CLASSES[-1])
    caught += len(ps.check_summary(s, r.note, r.text_result)) > 0
print(f'Injected errors caught: {caught / len(test):.1%}')

#### The summary never uses the radiologist's report

In [ ]:
import inspect
args = inspect.signature(ps.make_summary).parameters
print('make_summary inputs:', list(args))
assert not any(a in args for a in ('findings', 'impression', 'report')), 'the report must never be an input'

### Optional: fluent rewrite with a small language model

A small instruction model (Flan-T5) rewrites the template into a smoother paragraph.

#### Guard: a rewrite may not add facts or change numbers

In [ ]:
VOCAB = sorted({*ps.SYMPTOMS, *ps.HISTORY, *ps.PURPOSE, *CLASSES, *[ps.pretty(c) for c in CLASSES]} - {''}, key=len, reverse=True)
def facts_in(text, extra=()):
    t = ' ' + text.lower() + ' '
    return {v for v in [*VOCAB, *extra] if re.search(r'(?<![a-z])' + re.escape(v.lower()) + r'(?![a-z])', t)}
def numbers_in(text):
    return set(re.findall(r'\d+%', text))
def faithful(template, rewrite, terms=()):
    if not rewrite or len(rewrite.split()) < 0.4 * len(template.split()):
        return False, 'too short'
    added = facts_in(rewrite) - facts_in(template)
    if added:
        return False, f'added {sorted(added)}'
    dropped = facts_in(template, terms) - facts_in(rewrite, terms)
    if dropped:
        return False, f'dropped {sorted(dropped)}'
    if numbers_in(rewrite) != numbers_in(template):
        return False, 'changed or dropped a percentage'
    return True, 'ok'

#### Rewrite and check

In [ ]:
llm_rows = []
if RUN_LLM:
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        ltok = AutoTokenizer.from_pretrained(LLM_NAME)
        llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(dev).eval()
        sample = test[test.note != ''].sample(min(N_LLM, (test.note != '').sum()), random_state=SEED)
        for r in sample.itertuples():
            template = ' '.join(v for k, v in r.summary['sections'].items() if k != 'disclaimer')
            prompt = 'Rewrite this clinical summary as one short, clear paragraph for a doctor. Do not add any information.\n\n' + template
            enc = ltok(prompt, return_tensors='pt', truncation=True, max_length=512).to(dev)
            with torch.no_grad():
                out = llm.generate(**enc, max_new_tokens=160, num_beams=4)
            rewrite = ltok.decode(out[0], skip_special_tokens=True)
            terms = [t for k in KEYS + ['other'] for t in r.summary['facts']['note_terms'][k]]
            ok, why = faithful(template, rewrite, terms)
            llm_rows.append({'uid': r.uid, 'template': template, 'rewrite': rewrite, 'accepted': ok, 'reason': why})
    except Exception as e:
        print('Language model skipped:', e)

In [ ]:
llm_df = pd.DataFrame(llm_rows)
if len(llm_df):
    print(f'Rewrites accepted by the guard: {llm_df.accepted.mean():.0%} of {len(llm_df)}')
    display(llm_df.head(6)[['template', 'rewrite', 'accepted', 'reason']])

### Team rating sheet

Automatic checks prove the summaries are faithful, not that they are useful.

In [ ]:
sheet = test[test.note != ''].sample(min(N_RATING, (test.note != '').sum()), random_state=SEED)[['uid', 'note', 'summary_text']]
for who in ['rater_1', 'rater_2', 'rater_3']:
    for crit in ['accuracy', 'completeness', 'clarity']:
        sheet[f'{who}_{crit}'] = ''
sheet['comments'] = ''
sheet.to_csv(f'{OUT_DIR}/summary_rating_sheet.csv', index=False)
print('Saved summary_rating_sheet.csv with', len(sheet), 'summaries')

### Outputs

#### Save

In [ ]:
test[['uid', 'note', 'summary_text', 'problems']].to_csv(f'{OUT_DIR}/test_summaries.csv', index=False)
comparison.to_csv(f'{OUT_DIR}/extractor_comparison.csv')
coverage.to_csv(f'{OUT_DIR}/extractor_test_coverage.csv')
details[BEST].to_csv(f'{OUT_DIR}/extractor_gold_results.csv', index=False)
if independent is not None: independent.to_csv(f'{OUT_DIR}/extractor_independent_test.csv')
if len(llm_df): llm_df.to_csv(f'{OUT_DIR}/llm_rewrites.csv', index=False)
if os.path.abspath('pulmo_summary.py') != os.path.abspath(f'{OUT_DIR}/pulmo_summary.py'):
    shutil.copy('pulmo_summary.py', OUT_DIR)
with zipfile.ZipFile(f'{OUT_DIR}/summary_outputs.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    files = ['pulmo_summary.py', 'test_summaries.csv', 'summary_rating_sheet.csv', 'extractor_annotation_sheet.csv']
    files += ['extractor_comparison.csv', 'extractor_test_coverage.csv', 'extractor_gold_results.csv']
    files += ['extractor_independent_test.csv'] if independent is not None else []
    files += ['llm_rewrites.csv'] if len(llm_df) else []
    for f in files:
        z.write(f'{OUT_DIR}/{f}', f)
    for f in glob.glob(f'{OUT_DIR}/figures/*.png'): z.write(f, os.path.relpath(f, OUT_DIR))

In [ ]:
print('Saved summary_outputs.zip')

#### Final checks

In [ ]:
assert comparison.loc[BEST, 'F1'] >= 0.8, "extractor below 80% F1 on the gold notes"
assert (test.problems.str.len() == 0).all(), 'some summaries are not traceable'
assert 'Priority' in ps.make_summary('', {'has_note': False})['text']
print('All checks passed.')